In [1]:
!pip install dagshub mlflow statsforecast -q

import pandas as pd
import numpy as np
import dagshub
import mlflow
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
import warnings
warnings.filterwarnings('ignore')

REPO_OWNER = "ejoba22"  
REPO_NAME = "walmart-sales-forecasting"

dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)

mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 84.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 78.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=45294ed1-2986-4ef1-96e4-84dcb352d713&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=b6c91b0109862487eae9c01e77647180a78cb2ae7497d2bb7666047f31b2d37a




Accessing as tvada22

Initialized MLflow to track repo "ejoba22/walmart-sales-forecasting"

Repository ejoba22/walmart-sales-forecasting initialized!

In [2]:
# 2. DATA LOADING & FORMATTING
DATA_BASE_PATH = '/kaggle/input/notebooks/elenejobava/walmart-eda-feature-engineering/'
train_fe = pd.read_parquet(DATA_BASE_PATH + 'train_features.parquet')

# Clean and format for Nixtla (Same as Deep Learning)
if 'Type' in train_fe.columns:
    train_fe = train_fe.drop(columns=['Type'])

df_stats = train_fe.copy()
df_stats = df_stats.rename(columns={'Store_Dept': 'unique_id', 'Date': 'ds', 'Weekly_Sales': 'y'})
df_stats['weight'] = np.where(df_stats['IsHoliday'] == 1, 5, 1)

df_stats = df_stats.fillna(0)
df_stats = df_stats.sort_values(['unique_id', 'ds']).reset_index(drop=True)

# Time-Series Split (12 weeks)
split_date = df_stats['ds'].max() - pd.Timedelta(weeks=12)
train_df = df_stats[df_stats['ds'] <= split_date]
val_df = df_stats[df_stats['ds'] > split_date]

# Filter out short time series (AutoARIMA needs history to find seasonality)
series_lengths = train_df['unique_id'].value_counts()
valid_ids = series_lengths[series_lengths >= 52].index
train_df = train_df[train_df['unique_id'].isin(valid_ids)]
val_df = val_df[val_df['unique_id'].isin(valid_ids)]


In [3]:
# 3. METRICS
def calculate_wape(y_true, y_pred):
    return (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100

def calculate_wmae(y_true, y_pred, weights):
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [4]:
# 4. ARCHITECTURE SETUP & TRAINING

# Define the AutoARIMA model
models = [
    AutoARIMA(season_length=52) # 52 weeks in a year forces the S in SARIMA
]

# Initialize StatsForecast
sf = StatsForecast(
    models=models,
    freq='W-FRI',   # Friday alignment
    n_jobs=-1       # Use all Kaggle CPU cores for speed
)


# Pass strictly the 3 required columns so it doesn't ask for future feature values
train_df_arima = train_df[['unique_id', 'ds', 'y']]

# fit_predict automatically trains the math and generates the 12-week horizon
val_preds_df = sf.fit_predict(df=train_df_arima, h=12).reset_index()

# Merge predictions back with the original val_df (which still has our 'weight' column)
results = val_df[['unique_id', 'ds', 'y', 'weight']].merge(val_preds_df, on=['unique_id', 'ds'], how='inner')

In [6]:
# 5. MLFLOW LOGGING
mlflow.set_experiment("AutoARIMA_Training")

with mlflow.start_run(run_name="AutoARIMA_Baseline"):
    wmae_arima = calculate_wmae(results['y'], results['AutoARIMA'], results['weight'])
    wape_arima = calculate_wape(results['y'], results['AutoARIMA'])
    
    mlflow.log_param("architecture", "SARIMA")
    mlflow.log_param("season_length", 52)
    mlflow.log_metric("validation_WMAE", wmae_arima)
    mlflow.log_metric("validation_WAPE_percent", wape_arima)
    
    print(f"Validation WMAE: {wmae_arima:.2f}")
    print(f"Human-Readable Error (WAPE): {wape_arima:.2f}%")

Validation WMAE: 1513.97
Human-Readable Error (WAPE): 9.54%
🏃 View run AutoARIMA_Baseline at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/13/runs/6813d113de8f45a99a7a62b5d720d172
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/13
